# 03 - Latent space interpretation

What each block of the disentangled latent space encodes, on held-out samples of one fold.

In [ ]:
# parameters
RESULTS_DIR = "results"
DATA_DIR = "data/processed"
DATASET = "sim_c05"
VARIANT = "supervised_nocond"
BASELINE = "baseline"
FOLD = 0

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from mosa_disentangle import plotting
from mosa_disentangle.data import read_dataset
from mosa_disentangle.training import block_columns

sns.set_context("paper")
plt.rcParams["figure.dpi"] = 110

In [ ]:
def load_latent(variant):
    path = Path(RESULTS_DIR) / "runs" / DATASET / variant / f"fold{FOLD}" / "latent_val.parquet"
    return pd.read_parquet(path)

latent = load_latent(VARIANT)
blocks = sorted({c.rsplit("_", 1)[0] for c in latent.columns if c.rsplit("_", 1)[0] in
                 {"model_type", "tissue", "emt", "free"}})
latent_cols = {b: block_columns(latent.columns, b) for b in blocks}
print({b: len(c) for b, c in latent_cols.items()})
matrix = pd.read_csv(Path(RESULTS_DIR) / "runs" / DATASET / VARIANT / f"fold{FOLD}" / "probe_matrix.csv", index_col=0)
plotting.probe_heatmap(matrix, title=f"{DATASET} / {VARIANT}: held-out probes")

## Each block coloured by each factor

Rows: latent blocks (2-D PCA). Columns: colouring factor. A disentangled layout shows structure only on the diagonal.

In [ ]:
factors = [("model_type", False), ("tissue", False), ("emt", True)]
fig, axes = plt.subplots(len(blocks), len(factors), figsize=(5.5 * len(factors), 4 * len(blocks)))
for i, block in enumerate(blocks):
    values = latent[latent_cols[block]].to_numpy()
    if values.shape[1] < 2:
        values = np.column_stack([values, np.zeros(len(values))])
    for j, (factor, cont) in enumerate(factors):
        plotting.pca_scatter(values, latent[factor], ax=axes[i, j], continuous=cont,
                             title=f"{block} block, coloured by {factor}")
plt.tight_layout()

## Baseline for comparison

The same held-out samples in the unstructured baseline latent space.

In [ ]:
base = load_latent(BASELINE)
base_cols = block_columns(base.columns, "free")
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (factor, cont) in zip(axes, factors):
    plotting.pca_scatter(base[base_cols].to_numpy(), base[factor], ax=ax, continuous=cont,
                         title=f"baseline latent, coloured by {factor}")
plt.tight_layout()

## EMT block against marker genes

Correlation of each EMT-block coordinate with the EMT marker genes used to define the score, and with other genes, to check the block is reading the expected programme rather than a tissue proxy.

In [ ]:
from mosa_disentangle.data import EPITHELIAL_GENES, MESENCHYMAL_GENES

data = read_dataset(Path(DATA_DIR) / f"{DATASET}.h5mu", ["transcriptomics"])
expr = pd.DataFrame(data.views["transcriptomics"], index=data.metadata.index,
                    columns=data.feature_names["transcriptomics"])
shared = latent.index.intersection(expr.index)
markers = [g for g in EPITHELIAL_GENES + MESENCHYMAL_GENES if g in expr.columns]
if "emt" in latent_cols and markers:
    corr = pd.DataFrame({col: expr.loc[shared, markers].corrwith(latent.loc[shared, col])
                         for col in latent_cols["emt"]})
    corr["type"] = ["epithelial" if g in EPITHELIAL_GENES else "mesenchymal" for g in corr.index]
    display(corr.sort_values(latent_cols["emt"][0]).round(2))
    genome_wide = expr.loc[shared].corrwith(latent.loc[shared, latent_cols["emt"][0]])
    print("Top genes correlated with emt_0:")
    display(genome_wide.abs().sort_values(ascending=False).head(15).to_frame("|r|"))

## Tissue block

Hierarchical view of tissue centroids in the tissue block: related tissues should sit close together.

In [ ]:
if "tissue" in latent_cols:
    centroids = latent.groupby("tissue")[latent_cols["tissue"]].mean()
    counts = latent["tissue"].value_counts()
    centroids = centroids.loc[counts[counts >= 5].index]
    g = sns.clustermap(centroids.T.corr(), cmap="vlag", center=0, figsize=(7, 7))
    g.fig.suptitle("Tissue centroid correlation in the tissue block", y=1.02)